# Simple reusable preprocess functions

This notebook uses only string paths and `glob`.

It extracts:

- `workers`
- `processes = workers + 1`
- `cores`
- `time_s`
- `bab_nodes`

Output:

- `preprocessed_core_data.csv`


In [1]:
import glob
import json
import pandas as pd


## Text parser


In [28]:
def parse_text_file(filename):
    result = {
        "source": filename.split('/')[0],
        "workers": None,
        "cores": None,
        "time_s": None,
        "bab_nodes": None,
    }

    with open(filename, "r") as f:
        for line in f.readlines():
            if "Nodes =" in line:
                result["bab_nodes"] = int(line.strip().split(" ")[-1])

            if "Time =" in line:
                result["time_s"] = float(line.strip().split(" ")[-2])

            if "Maximum number of workers used:" in line:
                result["workers"] = int(line.strip().split(" ")[-1])

            if "Number of cores:" in line:
                result["cores"] = int(line.strip().split(" ")[-1])

    if result["workers"] is not None:
        result["cores"] = result["workers"] + 1

    return result


## JSON parser


In [29]:
def parse_json_file(filename):
    rows = []

    with open(filename, "r") as f:
        data = json.load(f)

    if not isinstance(data, list):
        data = [data]

    for item in data:
        meta = item["meta_data"]
        workers = meta["num_workers_used"]

        rows.append({
            "source": filename.split('/')[0],
            "workers": workers,
            "cores": workers + 1,
            "time_s": meta["time"],
            "bab_nodes": meta["eval_bab_nodes"],
        })

    return rows


## Load files with glob


In [30]:
def load_results(patterns):
    rows = []

    if isinstance(patterns, str):
        patterns = [patterns]

    filenames = []
    for pattern in patterns:
        filenames.extend(glob.glob(pattern))

    filenames = sorted(filenames)

    for filename in filenames:
        lower = filename.lower()
        if lower.endswith(".json"):
            rows.extend(parse_json_file(filename))
        else:
            rows.append(parse_text_file(filename))

    return rows


## Convert to clean DataFrame


In [31]:
def results_to_dataframe(rows):
    df = pd.DataFrame(rows)

    df = df.dropna(subset=["workers", "cores", "time_s", "bab_nodes"])
    df["cores"] = df["cores"].astype("Int64")
    df["workers"] = df["workers"].astype(int)
    df["time_s"] = df["time_s"].astype(float)
    df["bab_nodes"] = df["bab_nodes"].astype(int)        

    df = df.sort_values(["cores", "time_s"]).reset_index(drop=True)

    return df


## Use it here


In [32]:
patterns = ["org_solutions/*", "new_solutions/*", "newest_solutions/*"]

rows = load_results(patterns)
df = results_to_dataframe(rows)

display(df)


,source,workers,cores,time_s,bab_nodes
0,new_solutions,3,4,1897.596487,17781
1,org_solutions,3,4,2688.610000,17759
2,new_solutions,7,8,1095.912695,17783
3,org_solutions,7,8,1205.930000,17773
4,newest_solutions,7,8,1234.405245,17757
5,org_solutions,15,16,568.950000,17789
6,new_solutions,15,16,589.896755,17811
7,newest_solutions,15,16,592.235753,17811
8,new_solutions,31,32,253.532022,17795
9,org_solutions,31,32,276.850000,17743


## Save CSV


In [38]:
output_file = "preprocessed_data_org.csv"

df[df['source'] == 'org_solutions'].to_csv("preprocessed_data_org.csv", index=False)
df[df['source'] == 'new_solutions'].to_csv("preprocessed_data_new.csv", index=False)
df[df['source'] == 'newest_solutions'].to_csv("preprocessed_data_newest.csv", index=False)



## Quick summary


In [34]:
summary = (
    df.groupby(["workers", "cores"], as_index=False)
    .agg(
        runs=("time_s", "size"),
        mean_time_s=("time_s", "mean"),
        min_time_s=("time_s", "min"),
        max_time_s=("time_s", "max"),
        mean_bab_nodes=("bab_nodes", "mean"),
    )
)

display(summary)


,workers,cores,runs,mean_time_s,min_time_s,max_time_s,mean_bab_nodes
0,3,4,2,2293.103244,1897.596487,2688.610000,17770.000000
1,7,8,3,1178.749313,1095.912695,1234.405245,17771.000000
2,15,16,3,583.694169,568.950000,592.235753,17803.666667
3,31,32,2,265.191011,253.532022,276.850000,17769.000000
4,63,64,3,144.344166,141.176113,147.406386,17746.333333
5,127,128,3,75.516569,73.450000,77.249786,17750.333333
6,255,256,3,44.360342,42.990000,46.394310,17810.333333
7,511,512,3,32.349092,27.958048,40.270000,17739.000000
8,1023,1024,3,21.690422,16.589059,29.480000,17806.333333
